<a href="https://colab.research.google.com/github/jintubhuyan-2000/Spatial-Gradient-of-Highway-Induced-Land-Cover-Change/blob/main/SECTION_4_3_%E2%80%94_BUILT_UP_EXPANSION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# SECTION 4.3 — BUILT-UP EXPANSION
# ============================================================

INPUT_CSV = Path("TZPR_NLP_BuiltUp_Statistics.csv")
OUTPUT_DIR = Path("BuiltUp_Analysis_4_3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"CSV not found: {INPUT_CSV.resolve()}\n"
        "Put TZPR_NLP_BuiltUp_Statistics.csv in the same folder "
        "as this script."
    )

df = pd.read_csv(INPUT_CSV)

print("=" * 80)
print("TZPR–NLP | SECTION 4.3 BUILT-UP EXPANSION")
print("=" * 80)
print("\nColumns:", df.columns.tolist())
print("\nInput data:")
print(df.to_string(index=False))

# Standardize column names
df.columns = (
    df.columns.astype(str).str.strip().str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

# Detect class and area columns
class_candidates = ["class", "name", "lulc_class", "category"]
area_candidates = ["area_ha", "area", "hectares"]

class_col = next((c for c in class_candidates if c in df.columns), None)
area_col = next((c for c in area_candidates if c in df.columns), None)

if class_col is None or area_col is None:
    raise ValueError(
        f"Could not identify required columns. Available columns: {df.columns.tolist()}"
    )

df[class_col] = df[class_col].astype(str).str.strip()
df[area_col] = pd.to_numeric(df[area_col], errors="coerce")
df = df.dropna(subset=[area_col]).copy()

# Find GEE records
def get_value(labels):
    for label in labels:
        mask = df[class_col].str.lower().eq(label.lower())
        if mask.any():
            return float(df.loc[mask, area_col].iloc[0])
    return np.nan

built_2016 = get_value([
    "Built_2016",
    "Built-up_2016",
    "BuiltUp_2016"
])

built_2025 = get_value([
    "Built_2025",
    "Built-up_2025",
    "BuiltUp_2025"
])

new_built = get_value([
    "New_Built_2016_2025",
    "New_Built-up_2016_2025",
    "New_BuiltUp_2016_2025"
])

if pd.isna(built_2016) or pd.isna(built_2025):
    print("\nAvailable class labels:")
    print(df[class_col].tolist())
    raise ValueError("Built_2016 and/or Built_2025 were not found.")

# Core statistics
absolute_change = built_2025 - built_2016
percentage_change = absolute_change / built_2016 * 100 if built_2016 else np.nan
annual_change = absolute_change / 9.0

built_2016_km2 = built_2016 / 100
built_2025_km2 = built_2025 / 100
absolute_change_km2 = absolute_change / 100

if pd.isna(new_built):
    new_built_km2 = np.nan
    new_built_share = np.nan
else:
    new_built_km2 = new_built / 100
    new_built_share = new_built / built_2025 * 100 if built_2025 else np.nan

growth_factor = built_2025 / built_2016 if built_2016 else np.nan

# Manuscript summary
summary = pd.DataFrame([
    ["Built-up area, 2016", built_2016, built_2016_km2, "ha / km²"],
    ["Built-up area, 2025", built_2025, built_2025_km2, "ha / km²"],
    ["Absolute built-up expansion, 2016–2025",
     absolute_change, absolute_change_km2, "ha / km²"],
    ["Percentage increase in built-up area",
     np.nan, percentage_change, "%"],
    ["Average annual built-up expansion",
     annual_change, annual_change / 100, "ha/year"],
    ["New built-up area, 2016–2025",
     new_built, new_built_km2, "ha / km²"],
    ["New built-up area as % of 2025 built-up",
     np.nan, new_built_share, "%"],
    ["Built-up growth factor (2025/2016)",
     np.nan, growth_factor, "times"],
], columns=["Indicator", "Value_ha", "Value_km2_or_percent", "Unit"])

summary.to_csv(
    OUTPUT_DIR / "01_BuiltUp_Manuscript_Summary.csv", index=False
)

change = pd.DataFrame([{
    "Start_Year": 2016,
    "End_Year": 2025,
    "BuiltUp_2016_ha": built_2016,
    "BuiltUp_2025_ha": built_2025,
    "Absolute_Change_ha": absolute_change,
    "Absolute_Change_km2": absolute_change_km2,
    "Percentage_Change": percentage_change,
    "Average_Annual_Change_ha": annual_change,
    "New_BuiltUp_2016_2025_ha": new_built
}])

change.to_csv(
    OUTPUT_DIR / "02_BuiltUp_Change_Statistics.csv", index=False
)

# Values for writing the manuscript
txt = f"""
SECTION 4.3 — BUILT-UP EXPANSION
================================

Built-up area in 2016: {built_2016:,.2f} ha ({built_2016_km2:,.2f} km²)
Built-up area in 2025: {built_2025:,.2f} ha ({built_2025_km2:,.2f} km²)

Absolute increase: {absolute_change:,.2f} ha ({absolute_change_km2:,.2f} km²)
Percentage increase: {percentage_change:,.2f} %
Average annual expansion: {annual_change:,.2f} ha/year

New built-up area, 2016–2025:
{new_built:,.2f} ha ({new_built_km2:,.2f} km²)

New built-up area as percentage of 2025 built-up:
{new_built_share:,.2f} %

Growth factor:
{growth_factor:,.2f} times

FIGURE 4
Spatial expansion of built-up land between 2016 and 2025.
Maps:
- TZPR_NLP_BuiltUp_2016
- TZPR_NLP_BuiltUp_2025
- TZPR_NLP_New_BuiltUp_2016_2025

FIGURE 5
Spatial distribution of non-built land converted to built-up land during 2016–2025.
Maps:
- TZPR_NLP_NonBuilt_to_Built_2016_2025
- TZPR_NLP_Cropland_to_Built_Component
- TZPR_NLP_Trees_to_Built_Component

NOTE:
Use Figure 4 to describe the actual spatial pattern. Do not claim
concentration near towns, settlements, road junctions, etc. unless
the maps visibly demonstrate it.
"""

(OUTPUT_DIR / "03_BuiltUp_Manuscript_Values.txt").write_text(txt, encoding="utf-8")

print("\n" + "=" * 80)
print("MANUSCRIPT-READY VALUES")
print("=" * 80)
print(f"Built-up 2016       : {built_2016:,.2f} ha")
print(f"Built-up 2025       : {built_2025:,.2f} ha")
print(f"Absolute increase   : {absolute_change:,.2f} ha")
print(f"Percentage increase : {percentage_change:,.2f} %")
print(f"Annual increase     : {annual_change:,.2f} ha/year")
print(f"New built-up        : {new_built:,.2f} ha")
print(f"Growth factor       : {growth_factor:,.2f} times")

print("\nCreated:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" ", p)
